# Chapter 2 - Working with Text

## Byte pair encoding

Byte pair encoding (BPE) was used to train LLMs such as GPT-2, GPT-3 and the
models used in ChatGPT.

In [1]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.14.0


In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

text = ("Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [4]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [5]:
text = "Akwirw ier"
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

strings = tokenizer.decode(integers)
print(strings)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


## Data Sampling with a sliding window

To train the LLM, we need to provide it with a series of words and then the target
word that it should predict, then we include that word in the series and take the
next word. This is called a sliding window.

To do this efficiently, we use PyTorch's built in `Dataset` and `DataLoader` classes
to load the data into tensors.

In [13]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride) -> None:
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt)

    # Uses a sliding window to chunk the text into overlapping
    # sequences of max_length
    for i in range(0, len(token_ids) - max_length, stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]

      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))

  # Total number of rows in the dataset
  def __len__(self):
      return len(self.input_ids)

  # Returns a single row from the dataset
  def __getitem__(self, idx):
      return self.input_ids[idx], self.target_ids[idx]

Now we'll use the `GPTDatasetV1` to load the inputs in batches via a PyTorch `DataLoader`.

In [14]:
# A data loader to generate batches with input-width pairs
def create_dataloader_v1(txt:str, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0) -> DataLoader:
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
  dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=shuffle,
    # True drops the last batch if it is shorter than the specified batch_size
    # to prevent loss spikes during training
    drop_last=drop_last,
    # The number of CPU processes to use for preprocessing
    num_workers=num_workers
  )

  return dataloader

In [15]:
with open("../../ch02/01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
